In [26]:
import pandas as pd
from pathlib import Path
import pickle
import numpy as np

df = pd.read_csv('datasets/kneemridataset/metadata.csv')


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 917 entries, 0 to 916
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   examId          917 non-null    int64
 1   seriesNo        917 non-null    int64
 2   aclDiagnosis    917 non-null    int64
 3   kneeLR          917 non-null    int64
 4   roiX            917 non-null    int64
 5   roiY            917 non-null    int64
 6   roiZ            917 non-null    int64
 7   roiHeight       917 non-null    int64
 8   roiWidth        917 non-null    int64
 9   roiDepth        917 non-null    int64
 10  volumeFilename  917 non-null    str  
dtypes: int64(10), str(1)
memory usage: 78.9 KB


In [23]:
df.head(10)

,examId,seriesNo,aclDiagnosis,kneeLR,roiX,roiY,roiZ,roiHeight,roiWidth,roiDepth,volumeFilename,file_exists
0,329637,8,0,1,139,184,14,74,72,3,329637-8.pck,True
1,390116,9,0,0,113,105,10,83,98,6,390116-9.pck,True
2,404663,8,1,1,120,117,15,101,115,2,404663-8.pck,True
3,406320,9,0,0,117,124,12,91,80,3,406320-9.pck,True
4,412857,8,0,1,122,105,14,83,98,4,412857-8.pck,True
5,412865,8,1,0,111,133,13,78,78,4,412865-8.pck,True
6,415102,9,0,0,116,117,11,67,68,2,415102-9.pck,True
7,425707,8,0,0,113,116,11,95,86,3,425707-8.pck,True
8,425713,8,0,0,108,128,13,81,83,2,425713-8.pck,True
9,437474,8,0,1,107,120,12,71,69,2,437474-8.pck,True


# Revisar valores únicos en el dataset y buscar duplicados

In [ ]:
# revisar si ExamnId + seiesNo es una clave única
duplicated_examId = df.duplicated(subset=['examId', 'seriesNo']).any()

# Revisar que todos los archivos existan

In [29]:
dataset_path = Path('datasets/kneemridataset')

files_found = {
    file.name: file 
    for file in dataset_path.rglob('*') 
    if file.is_file()
}

df['file_exists'] = df['volumeFilename'].isin(files_found.keys())

df['complete_route'] = df['volumeFilename'].map(files_found)

total_files = len(df)
existing_files = df['file_exists'].sum()
lacking_files = total_files - existing_files

print(f"Total de registros: {total_files}")
print(f"Archivos encontrados: {existing_files}")
print(f"Archivos FALTANTES: {lacking_files}")

if lacking_files > 0:
    print("\nArchivos no encontrados en ninguna subcarpeta:")
    print(df[~df['file_exists']]['volumeFilename'].tolist())

Total de registros: 917
Archivos encontrados: 736
Archivos FALTANTES: 181

Archivos no encontrados en ninguna subcarpeta:
['919803-5.pck', '919809-5.pck', '919812-5.pck', '919858-5.pck', '922655-5.pck', '922799-5.pck', '922927-5.pck', '923490-5.pck', '924901-5.pck', '924915-5.pck', '927081-5.pck', '927817-5.pck', '928454-5.pck', '929002-5.pck', '929004-5.pck', '929005-5.pck', '929011-5.pck', '929014-5.pck', '929017-5.pck', '929038-5.pck', '931418-5.pck', '931491-6.pck', '931506-5.pck', '931509-5.pck', '932249-5.pck', '932265-6.pck', '932294-5.pck', '932304-6.pck', '932305-5.pck', '932338-5.pck', '933325-5.pck', '934291-5.pck', '934306-6.pck', '934316-5.pck', '934433-5.pck', '935664-5.pck', '936150-5.pck', '936593-6.pck', '936646-5.pck', '936714-6.pck', '938663-5.pck', '942595-5.pck', '944779-5.pck', '944819-5.pck', '947642-5.pck', '947843-5.pck', '950624-5.pck', '950665-5.pck', '950719-5.pck', '951154-5.pck', '951237-5.pck', '953440-5.pck', '953493-5.pck', '953522-6.pck', '954247-5.pck

In [30]:
def get_pck_shape(route):
  try:
    # Si la ruta es nula (el archivo no existía), devolver None
    if pd.isna(route):
      return None
    
    with open(route, 'rb') as f:
      volume = pickle.load(f)
      
      if isinstance(volume, np.ndarray):
        return volume.shape
      else:
        return f"No es array, es: {type(volume)}"
  except Exception as e:
      return f"Error: {e}"

df['volume_shape'] = df['complete_route'].apply(get_pck_shape)

# Ahora revisamos si todas las formas son iguales
unique_shapes = df['volume_shape'].value_counts()

print("\n--- Resultados de las formas de los volúmenes ---")
print(unique_shapes)

# Comprobación final
if len(unique_shapes) == 1:
    print("\nTodos los archivos tienen exactamente la misma forma.")
else:
    print(f"\nHay {len(unique_shapes)} formas diferentes en el dataset.")


--- Resultados de las formas de los volúmenes ---
volume_shape
(30, 320, 320)    284
(32, 320, 320)    195
(34, 320, 320)     43
(33, 320, 320)     42
(28, 320, 320)     34
(31, 320, 320)     27
(27, 320, 320)     25
(35, 320, 320)     24
(29, 320, 320)     15
(25, 320, 320)     14
(26, 320, 320)      9
(24, 320, 320)      5
(36, 320, 320)      4
(22, 320, 320)      3
(23, 320, 320)      2
(37, 320, 320)      2
(21, 320, 320)      1
(25, 292, 320)      1
(38, 320, 320)      1
(28, 320, 308)      1
(30, 290, 320)      1
(45, 320, 320)      1
(30, 320, 300)      1
(30, 312, 320)      1
Name: count, dtype: int64

Hay 24 formas diferentes en el dataset.
